In [ ]:
import sys
import os
base_dir = r"C:\Users\acostamendoza\projects\distance-project"
data_dir = r"D:\data-distance-coding-project\processed-data"
sys.path.append(base_dir)
from src.data.RadialDevice import RadialDevice
from src.viz.SessionVisualizer import SessionVisualizer
from src.viz.CellVisualizer import CellVisualizer
from src.analysis.CellAnalysis import CellAnalyzer, SessionAnalyzer

import numpy as np
import matplotlib.pyplot as plt
from src.workflows import load_dataframe

%load_ext autoreload
%autoreload 2

mouse_name = 'SAM041'
record_date = '251128'
experiment_name = 'tiling_3_120um'

device = RadialDevice(mouse_name=mouse_name, record_date=record_date, experiment_name=experiment_name)
session_df = load_dataframe(mouse_name=mouse_name, record_date=record_date, experiment_name=experiment_name)
viz = SessionVisualizer(session_df, animal_id=mouse_name, experiment_name=experiment_name)

In [ ]:
def compute_empirical_tuning_2d(cell, train_trials):
    """Compute mean and variance at each (angle, distance) combination."""
    n_angles = len(device.angle_range)
    n_distances = 4

    means = np.zeros((n_angles, n_distances))
    variances = np.zeros((n_angles, n_distances))

    for d in range(n_distances):
        dist_trials = [t for t in train_trials if device.distance_sequence[t] == d]
        if len(dist_trials) > 0:
            trial_responses = np.array([
                device.get_repeat_trace(cell, t) for t in dist_trials
            ])
            means[:, d] = np.mean(trial_responses, axis=0)
            variances[:, d] = np.var(trial_responses, axis=0)

    return means, variances


def compute_all_neuron_stats_2d(cells_to_include, train_trials):
    """Compute 2D tuning for all neurons."""
    n_neurons = len(cells_to_include)
    n_angles = len(device.angle_range)

    neuron_means = np.zeros((n_neurons, n_angles, 4))
    neuron_variances = np.zeros((n_neurons, n_angles, 4))

    for i, cell in enumerate(cells_to_include):
        means, variances = compute_empirical_tuning_2d(cell, train_trials)
        neuron_means[i] = means
        neuron_variances[i] = variances

    return neuron_means, neuron_variances

In [ ]:
def log_likelihood_2d(response_vector, angle_idx, dist_idx, neuron_means, neuron_variances):
    """Gaussian log-likelihood for (angle, distance) combination."""
    expected = neuron_means[:, angle_idx, dist_idx]
    variances = np.maximum(neuron_variances[:, angle_idx, dist_idx], 1e-6)
    return -np.sum((response_vector - expected)**2 / variances)


def decode_2d(response_vector, neuron_means, neuron_variances):
    """Find (angle, distance) that maximizes likelihood."""
    n_angles = neuron_means.shape[1]
    n_distances = 4

    ll_grid = np.zeros((n_angles, n_distances))
    for a in range(n_angles):
        for d in range(n_distances):
            ll_grid[a, d] = log_likelihood_2d(response_vector, a, d,
                                               neuron_means, neuron_variances)

    best_a, best_d = np.unravel_index(np.argmax(ll_grid), ll_grid.shape)
    return device.angle_range[best_a], best_d


def get_test_trial_responses(cells_to_include, test_idx):
    """Get response vectors for all neurons for a single test trial."""
    return np.array([device.get_repeat_trace(cell, test_idx) for cell in cells_to_include])


def decode_trajectory_2d(cells_to_include, neuron_means, neuron_variances, test_idx):
    """Decode angle and distance at each timepoint."""
    test_responses = get_test_trial_responses(cells_to_include, test_idx)

    decoded_angles = []
    decoded_distances = []
    for i in range(test_responses.shape[1]):
        angle, dist = decode_2d(test_responses[:, i], neuron_means, neuron_variances)
        decoded_angles.append(angle)
        decoded_distances.append(dist)

    return decoded_angles, decoded_distances

In [ ]:
def evaluate_decoder(actual, decoded):
    """Compute metrics for continuous decoder."""
    errors = decoded - actual
    return {
        'mae': np.mean(np.abs(errors)),
        'rmse': np.sqrt(np.mean(errors**2)),
        'bias': np.mean(errors),
        'correlation': np.corrcoef(actual.flatten(), decoded.flatten())[0, 1],
        'std': np.std(errors),
    }


def print_metrics(metrics):
    print("=" * 40)
    print(f"  MAE:          {metrics['mae']:.2f}")
    print(f"  RMSE:         {metrics['rmse']:.2f}")
    print(f"  Correlation:  {metrics['correlation']:.3f}")
    print("=" * 40)

In [ ]:
# Select reliable cells
reliable_cells_df = viz.filter_by_reliability(session_df, reliable_type='reliable', threshold=0.85)
cells_to_use = reliable_cells_df['cell_id'].values
print(f'Using {len(cells_to_use)} cells')

# Get all non-control trials
all_trial_idxs = device.get_condition_trials(monocular=None, distance=None)

k_folds = 10
all_decoded_angles = []
all_decoded_distances = []
all_actual_distances = []

for fold in range(k_folds):
    print(f'Fold {fold + 1}/{k_folds}', end='\r')

    trial_idxs = np.random.permutation(all_trial_idxs)
    split_point = int(0.9 * len(trial_idxs))
    train_idxs = trial_idxs[:split_point]
    test_idxs = trial_idxs[split_point:]

    neuron_means_2d, neuron_variances_2d = compute_all_neuron_stats_2d(cells_to_use, train_idxs)

    for test_idx in test_idxs:
        angles, dists = decode_trajectory_2d(cells_to_use, neuron_means_2d, neuron_variances_2d, test_idx)
        all_decoded_angles.append(angles)
        all_decoded_distances.append(dists)
        all_actual_distances.append(device.distance_sequence[test_idx])

print(f'\nTotal trials: {len(all_decoded_angles)}')

In [ ]:
decoded_angles_array = np.array(all_decoded_angles)
decoded_distances_array = np.array(all_decoded_distances)
actual_distances_array = np.array(all_actual_distances)
actual_angles_array = np.tile(device.angle_range, (len(all_decoded_angles), 1))

# Angle metrics
print("\n=== ANGLE DECODING ===")
angle_metrics = evaluate_decoder(actual_angles_array, decoded_angles_array)
print_metrics(angle_metrics)

# Distance metrics
actual_dist_expanded = np.repeat(actual_distances_array[:, np.newaxis], decoded_distances_array.shape[1], axis=1)
distance_accuracy = np.mean(decoded_distances_array == actual_dist_expanded)
print(f"\n=== DISTANCE DECODING ===")
print(f"  Accuracy: {distance_accuracy:.1%} (chance = 25%)")

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(actual_dist_expanded.flatten(), decoded_distances_array.flatten())

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=[0, 1, 2, 3], yticklabels=[0, 1, 2, 3])
ax.set_xlabel('Decoded Distance')
ax.set_ylabel('Actual Distance')
ax.set_title(f'Distance Confusion Matrix ({distance_accuracy:.1%})')
plt.show()

In [ ]:
# Distance accuracy vs angle
angle_bins = device.angle_range
accuracy_per_angle = []

for i, angle in enumerate(angle_bins):
    # Get decoded distances at this angle position (column i across all trials)
    decoded_at_angle = decoded_distances_array[:, i]
    actual_at_angle = actual_dist_expanded[:, i]
    
    accuracy = np.mean(decoded_at_angle == actual_at_angle)
    accuracy_per_angle.append(accuracy)

accuracy_per_angle = np.array(accuracy_per_angle)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(angle_bins, accuracy_per_angle * 100, 'b-', linewidth=2)
ax.axhline(25, color='r', linestyle='--', label='Chance (25%)')
ax.fill_between(angle_bins, 25, accuracy_per_angle * 100, 
                where=(accuracy_per_angle * 100 > 25), alpha=0.3, color='blue')
ax.set_xlabel('Angle (°)')
ax.set_ylabel('Distance Accuracy (%)')
ax.set_title('Distance Decoding Accuracy vs Angular Position')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim([angle_bins.min(), angle_bins.max()])
plt.tight_layout()
plt.show()

print(f"Mean accuracy: {np.mean(accuracy_per_angle)*100:.1f}%")
print(f"Best angle: {angle_bins[np.argmax(accuracy_per_angle)]:.1f}° ({np.max(accuracy_per_angle)*100:.1f}%)")
print(f"Worst angle: {angle_bins[np.argmin(accuracy_per_angle)]:.1f}° ({np.min(accuracy_per_angle)*100:.1f}%)")